In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
import matplotlib.dates as mdates
from sklearn import datasets, linear_model
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.lines import Line2D
import statsmodels.api as sm
from scipy.stats import t
from scipy.optimize import minimize
import os
import cvxpy as cp
from tqdm import tqdm
import seaborn as sns
import torch

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import openpyxl

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.autograd import Function

from sklearn.preprocessing import StandardScaler

from functions import CVaRSolver, SPOPlus, compute_loss_normalization
import functions

# Data Preparation

Do following for different risk aversions beta:

for date in test_dates:

1. prepare train and test data (date)

2. prepare scenario matrix out of train data

3. specify constants for cvxpy (dimensions of scenario matrix, etc.)

4. precompute oracle solution w*(c)

5. Train VAR on train with custom loss function (combinations of MSE and DFL loss)

    Do so with every combination
    Save relevant metrics
    Estimate returns c_hat and w*(c_hat)

-> Agreggate metrics ofer the whole test period (backtesting period)

In [2]:
data_path = "../data/processed/"

return_data = pd.read_csv(data_path + "stationary_return_data_subset.csv")

In [3]:
# Pivot return data
return_matrix = return_data.pivot(index='date', columns='RIC', values='return')
display(return_matrix)

RIC,BRO.N,CTRA.N,CVS.N,FFIV.OQ,HSY.N,LLY.N,NEM.N,PCG.N,REGN.OQ,TYL.N
date,,,,,,,,,,
2000-01-31,-0.104405,-0.081712,-1.236885e-01,-0.175439,-0.105263,0.005639,-0.168367,7.012195e-02,-0.034314,-0.204545
2000-02-29,-0.035086,0.074857,1.788909e-03,-0.042553,0.039651,-0.107465,0.085890,-5.982906e-02,3.588832,0.228571
2000-03-31,0.172348,0.142292,7.321429e-02,-0.247222,0.109531,0.059937,0.015535,3.281437e-02,-0.476770,0.104651
2000-04-28,0.037157,0.027682,1.595897e-01,-0.310886,-0.069231,0.227183,0.044568,2.351190e-01,-0.033827,-0.094737
2000-05-31,0.165747,0.345877,1.398903e-11,-0.309237,0.148974,-0.011941,-0.016000,1.444134e-11,-0.286652,-0.255814
...,...,...,...,...,...,...,...,...,...,...
2023-09-29,-0.057490,-0.040440,7.135185e-02,-0.015398,-0.068789,-0.030801,-0.052920,-1.042945e-02,-0.004271,-0.030846
2023-10-31,-0.004152,0.016636,-3.132471e-03,-0.059265,-0.063625,0.031277,0.014073,1.053937e-02,-0.052335,-0.034288
2023-11-30,0.076635,-0.038423,-1.536009e-02,0.129296,0.009148,0.068968,0.083216,5.337423e-02,0.056316,0.096380


In [4]:
X, Y = functions.create_time_series_data_with_lags(return_matrix, max_lag=3)

print(X.shape)
print(Y.shape)

(286, 30)
(286, 10)


## XX

In [5]:
# Code it for the last day as test day

test_index = 1 # index of the test day, counting from the end of the dataset (1 means the last day, 2 means the second to last day, etc.)

alpha = 0.95 # CVaR confidence level (95% confidence level means we are looking at the worst 5% of cases)
beta = 0.08 # CVaR risk aversion (average loss in percentage of initial investment in the worst 5% of cases)

In [6]:
X_train = X[:-test_index] # take all rows up to the test_index last row for training
X_test = X[-test_index] # take the test_index last row for testing

Y_train = Y[:-test_index]
Y_test = Y[-test_index]

len(Y_train), len(Y_test)

(285, 10)

In [7]:
# Perform bootstrap for return scenarios on the training period

return_array = return_matrix[:-test_index].values

# Number of bootstrap scenarios
num_scenarios = 1000

# Set random seed for reproducibility
np.random.seed(42)

# Sample row indices with replacement
sample_indices = np.random.choice(return_array.shape[0], size=num_scenarios, replace=True)

# Generate bootstrapped scenario matrix
scenario_matrix = return_array[sample_indices, :]

loss_matrix = -scenario_matrix
# scenario_matrix (S scenarios × N assets)

S, N = scenario_matrix.shape

# Empirical mean from scenarios
mu = scenario_matrix.mean(axis=0)

print(S, N)

1000 10


In [8]:
# instantiate general CVaR problem solver

solver = CVaRSolver(
    loss_matrix=loss_matrix,
    N=N,
    S=S,
    alpha=alpha,
    beta=beta
)

In [9]:
# precompute oracle solutions for training
# for every sample in train set we need the exact solution

# Precompute oracle solutions
oracle_solutions = [
    solver.solve(mu).copy()
    for mu in tqdm(Y_train)
]

100%|██████████| 285/285 [00:35<00:00,  7.94it/s]


In [10]:
# Convert to tensor
oracle_tensor = torch.tensor(
    np.array(oracle_solutions),
    dtype=torch.float32
)

# Prepare dataset for pytorch training
# scale only features. Predict unscaled returns because scaling those would mislead subsequent optimization.
x_scaler = StandardScaler()
X_train = x_scaler.fit_transform(X_train)
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)

Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)

# Include in dataset — index alignment is preserved through shuffling
train_dataset = TensorDataset(
    X_train_tensor,
    Y_train_tensor,
    oracle_tensor
)

In [11]:
batch_size = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True        # TensorDataset shuffles all tensors together
)

In [12]:
class VARasNN(nn.Module):

    def __init__(self,input_dim,output_dim):

        super().__init__()

        self.linear = nn.Linear(
            input_dim,
            output_dim
        )

    def forward(self,x):

        return self.linear(x)

In [18]:
input_dim = X_train.shape[1]
output_dim = Y_train.shape[1]

model = VARasNN(input_dim, output_dim)

print(model)

VARasNN(
  (linear): Linear(in_features=30, out_features=10, bias=True)
)


In [19]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [15]:
spo_scale, mse_scale = compute_loss_normalization(model, train_loader, solver, criterion)

print(f"SPO+ scale: {spo_scale:.6f}")
print(f"MSE  scale: {mse_scale:.6f}")

100%|██████████| 72/72 [00:30<00:00,  2.36it/s]

SPO+ scale: 0.012942
MSE  scale: 0.367950


In [ ]:
gamma = 0.9  # 0 = full MSE, 1 = full SPO+

criterion = nn.MSELoss()

n_epochs = 5

for epoch in tqdm(range(n_epochs), desc="epochs"):

    model.train()
    epoch_loss = 0

    for X_batch, Y_batch, oracle_batch in tqdm(train_loader, desc="batches"):

        optimizer.zero_grad()

        predictions = model(X_batch)

        # SPO+ loss
        spo_losses = []
        for i in range(X_batch.shape[0]):
            loss_i = SPOPlus.apply(
                predictions[i],
                Y_batch[i],
                oracle_batch[i],
                solver
            )
            spo_losses.append(loss_i)
        spo_loss = torch.stack(spo_losses).mean()

        # MSE loss
        mse_loss = criterion(predictions, Y_batch)

        # combined loss
        loss = gamma * (spo_loss /spo_scale) + (1 - gamma) * (mse_loss / mse_scale)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1}: avg SPO+ loss = {avg_loss:.6f}")

epochs:  20%|██        | 1/5 [00:28<01:52, 28.13s/it]

Epoch 1: avg SPO+ loss = -0.042773


epochs:  40%|████      | 2/5 [01:03<01:36, 32.30s/it]

Epoch 2: avg SPO+ loss = -0.100900


epochs:  60%|██████    | 3/5 [01:37<01:06, 33.08s/it]

Epoch 3: avg SPO+ loss = -0.102900


epochs:  80%|████████  | 4/5 [02:10<00:33, 33.16s/it]

Epoch 4: avg SPO+ loss = -0.121373


epochs: 100%|██████████| 5/5 [02:40<00:00, 32.13s/it]

Epoch 5: avg SPO+ loss = -0.159507


In [20]:
gamma = 0.5  # 0 = full MSE, 1 = full SPO+

criterion = nn.MSELoss()

n_epochs = 5

for epoch in tqdm(range(n_epochs), desc="epochs"):

    model.train()
    epoch_spo_loss = 0
    epoch_mse_loss = 0
    epoch_combined_loss = 0

    for X_batch, Y_batch, oracle_batch in train_loader:

        optimizer.zero_grad()
        predictions = model(X_batch)

        spo_losses = []
        for i in range(X_batch.shape[0]):
            loss_i = SPOPlus.apply(
                predictions[i],
                Y_batch[i],
                oracle_batch[i],
                solver
            )
            spo_losses.append(loss_i)
        spo_loss = torch.stack(spo_losses).mean()
        mse_loss = criterion(predictions, Y_batch)
        loss = gamma * (spo_loss / spo_scale) + (1 - gamma) * (mse_loss / mse_scale)

        loss.backward()
        optimizer.step()

        epoch_spo_loss      += spo_loss.item()
        epoch_mse_loss      += mse_loss.item()
        epoch_combined_loss += loss.item()

    n = len(train_loader)
    print(
        f"Epoch {epoch+1}: "
        f"combined={epoch_combined_loss/n:.6f} | "
        f"SPO+={epoch_spo_loss/n:.6f} | "
        f"MSE={epoch_mse_loss/n:.6f}"
    )

epochs:  20%|██        | 1/5 [00:29<01:58, 29.72s/it]

Epoch 1: combined=-4.792610 | SPO+=-0.136903 | MSE=0.365481


epochs:  40%|████      | 2/5 [01:02<01:34, 31.44s/it]

Epoch 2: combined=-5.222658 | SPO+=-0.151939 | MSE=0.476503


epochs:  60%|██████    | 3/5 [01:34<01:03, 31.63s/it]

Epoch 3: combined=-5.785776 | SPO+=-0.171718 | MSE=0.624479


epochs:  80%|████████  | 4/5 [02:08<00:32, 32.85s/it]

Epoch 4: combined=-7.322350 | SPO+=-0.216679 | MSE=0.772009


epochs: 100%|██████████| 5/5 [02:46<00:00, 33.32s/it]

Epoch 5: combined=-8.053891 | SPO+=-0.241000 | MSE=0.925154
